@channel Please use these dataset files for your task.

The provided AFL player information and season statistics datasets require validation before they can be used for analysis. Your task is to identify and resolve data quality issues, clean both datasets, and merge them into a single analysis-ready dataset. Ensure your cleaning decisions are documented throughout the notebook.

Deliverables
- Data Quality Assessment Report.
- Cleaned players_info.csv.
- Cleaned seasonal_stats.csv.
- Merged dataset (merged_players.csv).
- Cleaning log describing the issues identified, how they were resolved, and the rationale for each fix.
- Validation report including:
    - Row counts before and after cleaning
    - Missing values handled
    - Duplicate records removed
    - Unmatched player_id values identified after merging


- Jupyter Notebook (.ipynb) containing all cleaning, validation, and merging steps.
- Observations or insights about the cleaned dataset.


## Loading the datasets

In [1]:
import pandas as pd

In [2]:
players = pd.read_csv("afl_players_info_raw.csv")
stats = pd.read_csv("afl_players_seasonal_stats_raw.csv")

C:\Users\ahmed\AppData\Local\Temp\ipykernel_20000\4276660287.py:2: DtypeWarning: Columns (0: player_id) have mixed types. Specify dtype option on import or set low_memory=False.
  stats = pd.read_csv("afl_players_seasonal_stats_raw.csv")


## Initial inspection

In [3]:
stats["player_id"].dtype

dtype('O')

In [4]:
stats["player_id"].head(20)

0     43261
1     43261
2     43261
3     43261
4     43262
5     43262
6     43261
7     43261
8     43261
9     43261
10    43262
11    43262
12    43262
13    43262
14    43262
15    43262
16    43262
17    43262
18    43262
19    43263
Name: player_id, dtype: object

In [5]:
stats["player_id"].apply(type).value_counts()

player_id
<class 'int'>    16384
<class 'str'>     9107
Name: count, dtype: int64

In [6]:
stats["player_id"].unique()[:20]

array([43261, 43262, 43263, 45805, 43265, 43266, 45479, 43267, 43268,
       43269, 43346, 43270, 43271, 43273, 45896, 43274, 43275, 43276,
       43277, 43278], dtype=object)

In [7]:
stats["player_id"].isnull().sum()

np.int64(0)

In [8]:
stats["player_id"].describe()

count     25491
unique     3298
top       45522
freq         35
Name: player_id, dtype: object

In [9]:
players.info()

stats.info()

<class 'pandas.DataFrame'>
RangeIndex: 2848 entries, 0 to 2847
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   id                   2848 non-null   int64
 1   player_name          2848 non-null   str  
 2   player_full_name     2848 non-null   str  
 3   first_name           2848 non-null   str  
 4   last_name            2848 non-null   str  
 5   born_date            2848 non-null   str  
 6   debut_date           2848 non-null   str  
 7   debut_age            2848 non-null   int64
 8   last_date            2848 non-null   str  
 9   last_age             2848 non-null   int64
 10  height               2848 non-null   int64
 11  weight               2848 non-null   int64
 12  profile_pic          637 non-null    str  
 13  player_link          2848 non-null   str  
 14  player_common_names  75 non-null     str  
 15  player_teams         2754 non-null   str  
dtypes: int64(5), str(11)
memory usage: 

In [10]:
players.describe(include="all")

,id,player_name,player_full_name,first_name,last_name,born_date,debut_date,debut_age,last_date,last_age,height,weight,profile_pic,player_link,player_common_names,player_teams
count,2848.000000,2848,2848,2848,2848,2848,2848,2848.000000,2848,2848.000000,2848.000000,2848.000000,637,2848,75,2754
unique,NaN,2826,2843,742,1998,2546,1610,NaN,1305,NaN,NaN,NaN,636,2843,75,309
top,NaN,Ryan Abbott,Ryan_Abbott,Tom,Smith,1989-10-09,2012-03-24,NaN,2025-08-22,NaN,NaN,NaN,https://res.cloudinary.com/dijzdikkh/image/upl...,https://afltables.com/afl/stats/players/R/Ryan...,{Big Mac},{Essendon Bombers}
freq,NaN,2,2,63,24,4,15,NaN,67,NaN,NaN,NaN,2,2,1,131
mean,44858.623947,NaN,NaN,NaN,NaN,NaN,NaN,19.595857,NaN,25.765449,187.344803,86.012289,NaN,NaN,NaN,NaN
std,912.449281,NaN,NaN,NaN,NaN,NaN,NaN,1.824956,NaN,4.339853,7.318328,8.923524,NaN,NaN,NaN,NaN
min,43260.000000,NaN,NaN,NaN,NaN,NaN,NaN,16.000000,NaN,18.000000,163.000000,0.000000,NaN,NaN,NaN,NaN
25%,44063.750000,NaN,NaN,NaN,NaN,NaN,NaN,18.000000,NaN,22.000000,182.000000,80.000000,NaN,NaN,NaN,NaN
50%,44885.500000,NaN,NaN,NaN,NaN,NaN,NaN,19.000000,NaN,25.000000,187.000000,85.000000,NaN,NaN,NaN,NaN
75%,45662.250000,NaN,NaN,NaN,NaN,NaN,NaN,20.000000,NaN,29.000000,192.000000,92.000000,NaN,NaN,NaN,NaN


In [11]:
stats.describe(include="all")

,player_id,year,team,is_finals,games_played,kicks,marks,handballs,disposals,goals,...,avg_contested_possessions,avg_uncontested_possessions,avg_contested_marks,avg_marks_inside_50,avg_one_percenters,avg_bounces,avg_goal_assists,avg_score,avg_fantasy_points,avg_percentage_played
count,25491,25491.000000,25491,25491,25491.000000,25439.000000,25306.000000,25378.000000,25467.000000,22954.000000,...,22155.000000,22169.000000,20566.000000,20326.000000,21848.000000,19837.000000,18399.000000,25491.000000,25491.000000,19245.000000
unique,3298,NaN,114,2,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,45522,NaN,Geelong Cats,False,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,35,NaN,1263,19520,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,2010.194500,NaN,NaN,10.748617,98.697001,42.882281,67.689652,166.041583,7.169208,...,5.429564,8.575497,0.683696,0.732692,2.102938,0.665075,0.481434,3.591742,59.519850,77.742931
std,NaN,9.342562,NaN,NaN,7.832753,92.259369,39.119496,68.603571,154.153533,10.744221,...,2.585986,3.994015,0.672074,0.821606,1.629118,0.893630,0.487706,4.326204,21.492939,14.190194
min,NaN,1983.000000,NaN,NaN,-21.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-6.000000,0.000000
25%,NaN,2003.000000,NaN,NaN,3.000000,20.000000,9.000000,13.000000,34.000000,1.000000,...,3.800000,5.600000,0.100000,0.000000,1.000000,0.000000,0.000000,0.400000,45.000000,72.500000
50%,NaN,2011.000000,NaN,NaN,10.000000,66.000000,31.000000,44.000000,114.000000,3.000000,...,5.000000,8.000000,0.500000,0.500000,1.700000,0.300000,0.300000,2.300000,58.700000,80.400000
75%,NaN,2018.000000,NaN,NaN,19.000000,161.000000,70.000000,103.000000,268.000000,9.000000,...,6.600000,11.100000,1.000000,1.000000,2.600000,1.000000,0.800000,5.200000,73.800000,86.300000


In [12]:
players[["debut_age", "last_age", "height", "weight"]].describe()

,debut_age,last_age,height,weight
count,2848.000000,2848.000000,2848.000000,2848.000000
mean,19.595857,25.765449,187.344803,86.012289
std,1.824956,4.339853,7.318328,8.923524
min,16.000000,18.000000,163.000000,0.000000
25%,18.000000,22.000000,182.000000,80.000000
50%,19.000000,25.000000,187.000000,85.000000
75%,20.000000,29.000000,192.000000,92.000000
max,29.000000,40.000000,211.000000,118.000000


In [13]:
players.duplicated().sum()

np.int64(5)

In [14]:
stats.duplicated().sum()

np.int64(10)

In [15]:
players["id"].duplicated().sum()

np.int64(5)

In [16]:
players[players["weight"] == 0]

,id,player_name,player_full_name,first_name,last_name,born_date,debut_date,debut_age,last_date,last_age,height,weight,profile_pic,player_link,player_common_names,player_teams
1824,45612,Jack Hutchinson,Jack_Hutchinson,jack,HUTCHINSON,2001-11-10,2024-06-08,22,2025-07-25,23,190,0,https://res.cloudinary.com/dijzdikkh/image/upl...,https://afltables.com/afl/stats/players/J/Jack...,NaN,{West Coast Eagles}
2228,46122,Tom Hanily,Tom_Hanily,Tom,Hanily,2005-05-31,2025-03-06,19,2025-06-07,20,179,0,https://res.cloudinary.com/dijzdikkh/image/upl...,https://afltables.com/afl/stats/players/T/Tom_...,NaN,{Sydney Swans}


## Cleaning Datasets

### Step 1: Save row counts before cleaning

In [17]:
# Need these for validation report
players_before = len(players)
stats_before = len(stats)

print("Players before:", players_before)
print("Stats before:", stats_before)

Players before: 2848
Stats before: 25491


### Step 2: Clean duplicate rows

In [18]:
players = players.drop_duplicates()
stats = stats.drop_duplicates()

In [19]:
print(players.duplicated().sum())
print(stats.duplicated().sum())

0
0


### Step 3: Standardize the ID columns

In [20]:
# Rename the player dataset's ID column so both datasets use the same name.
players = players.rename(columns={"id": "player_id"})

# Convert both IDs to strings.
players["player_id"] = players["player_id"].astype(str)
stats["player_id"] = stats["player_id"].astype(str)

# Check
players["player_id"].dtype
stats["player_id"].dtype

<StringDtype(storage='python', na_value=nan)>

### Step 4: Fix invalid weights

In [21]:
# Replace 0 with missing values.
players["weight"] = players["weight"].replace(0, pd.NA)

# check
players["weight"].isna().sum()

np.int64(2)

In [22]:
# fill them with median
players["weight"] = players["weight"].fillna(players["weight"].median())

# check
players["weight"].min()

63

### Step 5: Convert date columns

In [23]:
date_cols = ["born_date", "debut_date", "last_date"]

for col in date_cols:
    players[col] = pd.to_datetime(players[col])

In [24]:
# Check
players.info()

<class 'pandas.DataFrame'>
Index: 2843 entries, 0 to 2847
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   player_id            2843 non-null   str           
 1   player_name          2843 non-null   str           
 2   player_full_name     2843 non-null   str           
 3   first_name           2843 non-null   str           
 4   last_name            2843 non-null   str           
 5   born_date            2843 non-null   datetime64[us]
 6   debut_date           2843 non-null   datetime64[us]
 7   debut_age            2843 non-null   int64         
 8   last_date            2843 non-null   datetime64[us]
 9   last_age             2843 non-null   int64         
 10  height               2843 non-null   int64         
 11  weight               2843 non-null   object        
 12  profile_pic          636 non-null    str           
 13  player_link          2843 non-null   str         

In [25]:
players["weight"] = pd.to_numeric(players["weight"])

In [26]:
players.info()

<class 'pandas.DataFrame'>
Index: 2843 entries, 0 to 2847
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   player_id            2843 non-null   str           
 1   player_name          2843 non-null   str           
 2   player_full_name     2843 non-null   str           
 3   first_name           2843 non-null   str           
 4   last_name            2843 non-null   str           
 5   born_date            2843 non-null   datetime64[us]
 6   debut_date           2843 non-null   datetime64[us]
 7   debut_age            2843 non-null   int64         
 8   last_date            2843 non-null   datetime64[us]
 9   last_age             2843 non-null   int64         
 10  height               2843 non-null   int64         
 11  weight               2843 non-null   int64         
 12  profile_pic          636 non-null    str           
 13  player_link          2843 non-null   str         

### Step 6: Verify duplicate IDs

In [27]:
players["player_id"].duplicated().sum()

np.int64(0)

### Step 7: Assess missing values in stats

In [28]:
# inspect
stats.isnull().sum().sort_values(ascending=False)

avg_goal_assists               7092
goal_assists                   7092
brownlow_votes                 6997
avg_percentage_played          6245
total_percentage_played        6245
hit_outs                       5706
avg_hit_outs                   5706
avg_bounces                    5654
bounces                        5654
marks_inside_50                5165
avg_marks_inside_50            5165
contested_marks                4925
avg_contested_marks            4925
avg_one_percenters             3643
one_percenters                 3643
rebound_50s                    3533
avg_rebound_50s                3533
avg_clearances                 3435
clearances                     3435
avg_contested_possessions      3336
contested_possessions          3336
uncontested_possessions        3322
avg_uncontested_possessions    3322
avg_inside_50s                 2971
inside_50s                     2971
avg_clangers                   2938
clangers                       2938
behinds                     

In [29]:
# calculate the percentage of missing values
missing = pd.DataFrame({
    "Missing Values": stats.isnull().sum(),
    "Percentage": (stats.isnull().sum() / len(stats) * 100).round(2)
})

missing.sort_values("Percentage", ascending=False)

,Missing Values,Percentage
avg_goal_assists,7092,27.83
goal_assists,7092,27.83
brownlow_votes,6997,27.46
avg_percentage_played,6245,24.51
total_percentage_played,6245,24.51
hit_outs,5706,22.39
avg_hit_outs,5706,22.39
avg_bounces,5654,22.19
bounces,5654,22.19
marks_inside_50,5165,20.27


### Step 8: Look for impossible numeric values

In [30]:
stats.describe()

,year,games_played,kicks,marks,handballs,disposals,goals,behinds,hit_outs,tackles,...,avg_contested_possessions,avg_uncontested_possessions,avg_contested_marks,avg_marks_inside_50,avg_one_percenters,avg_bounces,avg_goal_assists,avg_score,avg_fantasy_points,avg_percentage_played
count,25481.000000,25481.000000,25429.000000,25296.000000,25368.000000,25457.000000,22944.000000,22792.000000,19775.000000,24970.000000,...,22145.000000,22159.000000,20556.000000,20316.000000,21838.000000,19827.000000,18389.000000,25481.000000,25481.000000,19236.000000
mean,2010.192104,10.752914,98.717409,42.892078,67.699622,166.071768,7.169064,5.131801,21.320202,26.031197,...,5.430011,8.577034,0.683922,0.732703,2.103063,0.665300,0.481598,3.591288,59.522837,77.744521
std,9.342583,7.828029,92.264233,39.121944,68.609483,154.162626,10.745042,6.807211,80.565910,26.820492,...,2.586016,3.994072,0.672130,0.821743,1.629113,0.893751,0.487762,4.326688,21.492696,14.189090
min,1983.000000,-21.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-6.000000,0.000000
25%,2003.000000,3.000000,20.000000,9.000000,13.000000,34.000000,1.000000,1.000000,0.000000,6.000000,...,3.800000,5.700000,0.100000,0.000000,1.000000,0.000000,0.000000,0.400000,45.000000,72.500000
50%,2011.000000,10.000000,67.000000,31.000000,44.000000,114.000000,3.000000,3.000000,0.000000,17.000000,...,5.000000,8.000000,0.500000,0.500000,1.700000,0.300000,0.300000,2.300000,58.700000,80.400000
75%,2018.000000,19.000000,161.000000,70.000000,103.000000,268.000000,9.000000,7.000000,4.000000,38.000000,...,6.600000,11.100000,1.000000,1.000000,2.600000,1.000000,0.800000,5.200000,73.800000,86.300000
max,2025.000000,23.000000,512.000000,226.000000,482.000000,787.000000,123.000000,84.000000,1007.000000,205.000000,...,22.000000,29.000000,6.000000,8.000000,22.000000,13.000000,4.000000,59.000000,149.000000,100.000000


In [31]:
stats.describe().loc[["min", "max"]]

,year,games_played,kicks,marks,handballs,disposals,goals,behinds,hit_outs,tackles,...,avg_contested_possessions,avg_uncontested_possessions,avg_contested_marks,avg_marks_inside_50,avg_one_percenters,avg_bounces,avg_goal_assists,avg_score,avg_fantasy_points,avg_percentage_played
min,1983.0,-21.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-6.0,0.0
max,2025.0,23.0,512.0,226.0,482.0,787.0,123.0,84.0,1007.0,205.0,...,22.0,29.0,6.0,8.0,22.0,13.0,4.0,59.0,149.0,100.0


In [32]:
stats[stats["games_played"] < 0]

,player_id,year,team,is_finals,games_played,kicks,marks,handballs,disposals,goals,...,avg_contested_possessions,avg_uncontested_possessions,avg_contested_marks,avg_marks_inside_50,avg_one_percenters,avg_bounces,avg_goal_assists,avg_score,avg_fantasy_points,avg_percentage_played
19642,43260,2020,Richmond Tigers,False,-13,67.0,27.0,61.0,128.0,14.0,...,4.1,6.1,0.1,0.8,0.7,0.2,0.3,6.8,43.6,79.8
19643,43260,2020,Richmond Tigers,True,-1,5.0,4.0,3.0,8.0,0.0,...,3.0,5.0,0.0,0.0,1.0,0.0,0.0,0.0,45.0,78.0
25030,43260,2021,Richmond Tigers,False,-21,122.0,53.0,100.0,222.0,18.0,...,4.9,5.9,0.2,0.4,0.8,0.4,0.6,5.7,47.4,76.4
25031,43260,2022,Richmond Tigers,False,-7,21.0,7.0,29.0,50.0,2.0,...,3.6,3.6,0.0,0.4,0.6,0.0,0.1,2.0,28.4,38.9


In [33]:
stats[stats["year"] < 1900]

,player_id,year,team,is_finals,games_played,kicks,marks,handballs,disposals,goals,...,avg_contested_possessions,avg_uncontested_possessions,avg_contested_marks,avg_marks_inside_50,avg_one_percenters,avg_bounces,avg_goal_assists,avg_score,avg_fantasy_points,avg_percentage_played


In [34]:
stats[stats["games_played"] > 30]

,player_id,year,team,is_finals,games_played,kicks,marks,handballs,disposals,goals,...,avg_contested_possessions,avg_uncontested_possessions,avg_contested_marks,avg_marks_inside_50,avg_one_percenters,avg_bounces,avg_goal_assists,avg_score,avg_fantasy_points,avg_percentage_played


In [35]:
stats.loc[stats["games_played"] < 0, "games_played"] = pd.NA

In [36]:
stats["games_played"].isnull().sum()

np.int64(4)

In [37]:
stats[stats["avg_fantasy_points"] < 0]

,player_id,year,team,is_finals,games_played,kicks,marks,handballs,disposals,goals,...,avg_contested_possessions,avg_uncontested_possessions,avg_contested_marks,avg_marks_inside_50,avg_one_percenters,avg_bounces,avg_goal_assists,avg_score,avg_fantasy_points,avg_percentage_played
344,45896,1998,port adelaide power,False,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-3.0,NaN
3302,43812,2013,ST KILDA SAINTS,False,1.0,0.0,0.0,3.0,3.0,0.0,...,3.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,-6.0,50.0
7215,44401,2022,Western Bulldogs,True,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.0,4.0
10017,44846,2021,Richmond Tigers,False,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,-2.0,55.0
11860,45147,2019,North Melbourne Kangaroos,False,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,-1.0,13.0
15754,45632,1999,west coast eagles,True,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.0,NaN,NaN,0.0,-3.0,NaN
19511,46095,2013,St Kilda Saints,False,1.0,0.0,0.0,3.0,3.0,0.0,...,3.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,-6.0,50.0
21704,45744,2003,carlton blues,False,1.0,NaN,NaN,1.0,1.0,NaN,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-4.0,16.0
23614,45756,2007,WESTERN BULLDOGS,False,1.0,NaN,1.0,2.0,2.0,NaN,...,NaN,3.0,NaN,NaN,1.0,NaN,NaN,0.0,-1.0,32.0
23773,45945,2001,ESSENDON BOMBERS,False,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-2.0,NaN


In [38]:
stats["team"].sort_values().unique()

<StringArray>
[             ' ADELAIDE CROWS ',              ' Adelaide Crows ',
              ' BRISBANE LIONS ',              ' Brisbane Lions ',
               ' CARLTON BLUES ',         ' COLLINGWOOD MAGPIES ',
               ' Carlton Blues ',         ' Collingwood Magpies ',
            ' ESSENDON BOMBERS ',            ' Essendon Bombers ',
 ...
 'greater western sydney giants',                'hawthorn hawks',
              'melbourne demons',     'north melbourne kangaroos',
           'port adelaide power',               'richmond tigers',
               'st kilda saints',                  'sydney swans',
             'west coast eagles',              'western bulldogs']
Length: 114, dtype: str

In [39]:
# clean the team column
stats["team"] = stats["team"].str.strip()

In [40]:
# statndardize capitalization
stats["team"] = stats["team"].str.title()

In [41]:
# now we check again
stats["team"].nunique()

20

In [42]:
stats["team"].sort_values().unique()

<StringArray>
[               'Adelaide Crows',                'Brisbane Bears',
                'Brisbane Lions',                 'Carlton Blues',
           'Collingwood Magpies',              'Essendon Bombers',
                 'Fitzroy Lions',             'Fremantle Dockers',
                  'Geelong Cats',               'Gold Coast Suns',
 'Greater Western Sydney Giants',                'Hawthorn Hawks',
              'Melbourne Demons',     'North Melbourne Kangaroos',
           'Port Adelaide Power',               'Richmond Tigers',
               'St Kilda Saints',                  'Sydney Swans',
             'West Coast Eagles',              'Western Bulldogs']
Length: 20, dtype: str

In [43]:
stats["team"].value_counts().head(20)

team
Geelong Cats                     1689
West Coast Eagles                1661
Sydney Swans                     1620
Essendon Bombers                 1602
Collingwood Magpies              1576
Western Bulldogs                 1518
North Melbourne Kangaroos        1488
Adelaide Crows                   1483
Carlton Blues                    1457
Hawthorn Hawks                   1455
St Kilda Saints                  1437
Brisbane Lions                   1432
Melbourne Demons                 1395
Richmond Tigers                  1390
Port Adelaide Power              1355
Fremantle Dockers                1338
Greater Western Sydney Giants     755
Gold Coast Suns                   629
Brisbane Bears                    128
Fitzroy Lions                      73
Name: count, dtype: int64

## Saving the cleaned datasets

In [44]:
players.to_csv("cleaned_players_info.csv", index=False)

stats.to_csv("cleaned_seasonal_stats.csv", index=False)

## Merging cleaned Datasets into 1

In [45]:
# using outer join
merged_validation = pd.merge(
    players,
    stats,
    on="player_id",
    how="outer",
    indicator=True
)

In [46]:
# check
merged_validation["_merge"].value_counts()

_merge
both          25072
right_only      409
left_only         1
Name: count, dtype: int64

In [47]:
# identify unmatched IDS
players_only = merged_validation[merged_validation["_merge"] == "left_only"]
stats_only = merged_validation[merged_validation["_merge"] == "right_only"]

In [49]:
print("Player IDs only in players dataset:")
display(players_only[["player_id"]])

print("Player IDs only in stats dataset (first 10):")
display(stats_only[["player_id"]].head(10))

Player IDs only in players dataset:


,player_id
25220,46153


Player IDs only in stats dataset (first 10):


,player_id
191,43280
192,43280
288,43292
323,43299
474,43319
475,43319
519,43325
520,43325
540,43329
564,43332


In [50]:
# create final merged dataset
merged_players = pd.merge(
    players,
    stats,
    on="player_id",
    how="inner"
)

In [51]:
# verify
print(merged_players.shape)

(25072, 69)


In [52]:
# save it
merged_players.to_csv("merged_players.csv", index=False)

# Validation Report

### Row Counts Before and After Cleaning

| Dataset | Before Cleaning | After Cleaning |
|---------|----------------:|---------------:|
| Players | 2848 | 2843 |
| Seasonal Stats | 25491 | 25481 |

### Missing Values Handled

- Two invalid weight values (0 kg) in the players dataset were replaced with the median weight.
- Four negative values in the `games_played` column were treated as missing values (`NaN`).
- Existing missing statistical values were retained because they represent unavailable or unrecorded statistics rather than data entry errors.

### Duplicate Records Removed

- Players dataset: 5 duplicate records removed.
- Seasonal statistics dataset: 10 duplicate records removed.

## Merge Validation

- Successfully matched records: 25,072
- Player records without seasonal statistics: 1
- Seasonal statistics records without matching player information: 409

In [54]:
###############################################################################

# Cleaning Log

| Issue | Resolution | Reason |
|-------|------------|--------|
| Mixed data type in `player_id` | Converted to string | Ensured consistent merge key |
| Duplicate player records | Removed duplicates | Prevent duplicate entries |
| Duplicate seasonal statistics | Removed duplicates | Improve data integrity |
| Invalid weight values (0 kg) | Replaced with median weight | Zero is not a valid player weight |
| Negative values in `games_played` | Replaced with missing values (`NaN`) | Negative games played are impossible |
| Inconsistent team names | Removed extra spaces and standardized capitalization | Prevent duplicate team categories |
| Date columns | Converted to datetime format | Easier analysis and validation |

In [55]:
###############################################################################

# Data Quality Assessment Report

The datasets were inspected for common data quality issues before analysis.

The following checks were performed:

- Data types
- Missing values
- Duplicate records
- Invalid numeric values
- Inconsistent text formatting
- Merge key consistency (`player_id`)

Several issues were identified, including duplicate records, inconsistent team names, mixed data types for `player_id`, invalid weight values, and negative values in `games_played`. These issues were cleaned before merging the datasets.

In [56]:
###############################################################################

# Observations

- Most player records matched successfully with seasonal statistics.
- Only one player did not have matching seasonal statistics.
- 409 seasonal statistics records did not have corresponding player information.
- Team names required standardization due to inconsistent capitalization and whitespace.
- Many statistical columns contained missing values, which appear to represent unavailable data rather than data quality errors.
- The cleaned datasets are now suitable for further analysis.